In [1]:
DATE_START        = "2010-01-01"
DATE_END          = "2030-12-31"
dim_date_table    = "dim_date"
dim_zone_table    = "dim_zone"
write_mode        = "overwrite"

TLC_ZONE_CSV_URL  = "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv"

StatementMeta(, 55d2211c-e03c-4dd1-9832-482e11e34981, 3, Finished, Available, Finished, False)

In [2]:
import requests
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, BooleanType, ShortType, DateType
)

spark = SparkSession.builder.getOrCreate()

StatementMeta(, 55d2211c-e03c-4dd1-9832-482e11e34981, 4, Finished, Available, Finished, False)

In [3]:
print("Building DimDate …")

# Spark date_range via sequence + explode
date_range_df = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{DATE_START}'),
            to_date('{DATE_END}'),
            interval 1 day
        )
    ) AS full_date
""")

dim_date_df = (
    date_range_df
    .withColumn("date_key",      F.date_format("full_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("year",          F.year("full_date").cast(ShortType()))
    .withColumn("quarter",       F.quarter("full_date").cast(ShortType()))
    .withColumn("month",         F.month("full_date").cast(ShortType()))
    .withColumn("month_name",    F.date_format("full_date", "MMMM").cast(StringType()))
    .withColumn("month_abbr",    F.date_format("full_date", "MMM").cast(StringType()))
    .withColumn("week_of_year",  F.weekofyear("full_date").cast(ShortType()))
    .withColumn("day_of_month",  F.dayofmonth("full_date").cast(ShortType()))
    .withColumn("day_of_week",   F.dayofweek("full_date").cast(ShortType()))   # 1=Sun
    .withColumn("day_name",      F.date_format("full_date", "EEEE").cast(StringType()))
    .withColumn("day_abbr",      F.date_format("full_date", "EEE").cast(StringType()))
    .withColumn("is_weekend",
        (F.dayofweek("full_date").isin(1, 7)).cast(BooleanType()))
    .withColumn("is_leap_year",
        (
            (F.year("full_date") % 4 == 0)
            & ((F.year("full_date") % 100 != 0) | (F.year("full_date") % 400 == 0))
        ).cast(BooleanType())
    )
    .withColumn("year_month",    F.date_format("full_date", "yyyy-MM").cast(StringType()))
    .withColumn("year_quarter",
        F.concat(F.year("full_date").cast(StringType()),
                 F.lit("-Q"),
                 F.quarter("full_date").cast(StringType())).cast(StringType()))
    .select(
        "date_key", "full_date", "year", "quarter", "month", "month_name",
        "month_abbr", "week_of_year", "day_of_month", "day_of_week",
        "day_name", "day_abbr", "is_weekend", "is_leap_year",
        "year_month", "year_quarter",
    )
)

date_count = dim_date_df.count()
print(f"DimDate rows : {date_count:,}")

(
    dim_date_df.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(dim_date_table)
)
print(f"[OK] silver.{dim_date_table} written")

StatementMeta(, 55d2211c-e03c-4dd1-9832-482e11e34981, 5, Finished, Available, Finished, False)

Building DimDate …
DimDate rows : 7,670
[OK] silver.dim_date written


In [4]:
print("\nDownloading TLC zone lookup …")

zone_raw_path = "/lakehouse/default/Files/_reference/taxi_zone_lookup.csv"
os.makedirs(os.path.dirname(zone_raw_path), exist_ok=True)

r = requests.get(TLC_ZONE_CSV_URL, timeout=30)
r.raise_for_status()
with open(zone_raw_path, "wb") as f:
    f.write(r.content)

print(f"Downloaded {len(r.content):,} bytes → {zone_raw_path}")

StatementMeta(, 55d2211c-e03c-4dd1-9832-482e11e34981, 6, Finished, Available, Finished, False)


Downloaded 12,331 bytes → /lakehouse/default/Files/_reference/taxi_zone_lookup.csv


In [5]:
# TLC CSV columns: LocationID, Borough, Zone, service_zone
raw_zone_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/_reference/taxi_zone_lookup.csv")
)

dim_zone_df = (
    raw_zone_df
    .withColumnRenamed("LocationID",   "location_id")
    .withColumnRenamed("Borough",      "borough")
    .withColumnRenamed("Zone",         "zone_name")
    .withColumnRenamed("service_zone", "service_zone")
    .withColumn("location_id",  F.col("location_id").cast(IntegerType()))
    .withColumn("borough",      F.trim(F.col("borough").cast(StringType())))
    .withColumn("zone_name",    F.trim(F.col("zone_name").cast(StringType())))
    .withColumn("service_zone", F.trim(F.col("service_zone").cast(StringType())))
    # Derive a simple is_airport flag for analytical convenience
    .withColumn("is_airport",
        F.col("zone_name").rlike("(?i)(airport|JFK|LaGuardia|EWR)").cast(BooleanType()))
    .dropDuplicates(["location_id"])
    .orderBy("location_id")
)

zone_count = dim_zone_df.count()
print(f"DimZone rows : {zone_count}")

(
    dim_zone_df.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(dim_zone_table)
)
print(f"[OK] silver.{dim_zone_table} written")

StatementMeta(, 55d2211c-e03c-4dd1-9832-482e11e34981, 7, Finished, Available, Finished, False)

DimZone rows : 265
[OK] silver.dim_zone written


In [6]:
print("\n--- DimDate spot check ---")
spark.sql(f"""
    SELECT * FROM {dim_date_table}
    WHERE full_date IN ('2024-01-01','2024-07-04','2024-12-31')
""").show()

print("--- DimZone borough breakdown ---")
spark.sql(f"""
    SELECT borough, COUNT(*) AS zones,
           SUM(CAST(is_airport AS INT)) AS airport_zones
    FROM {dim_zone_table}
    GROUP BY 1
    ORDER BY 2 DESC
""").show()

StatementMeta(, 55d2211c-e03c-4dd1-9832-482e11e34981, 8, Finished, Available, Finished, False)


--- DimDate spot check ---
+--------+----------+----+-------+-----+----------+----------+------------+------------+-----------+--------+--------+----------+------------+----------+------------+
|date_key| full_date|year|quarter|month|month_name|month_abbr|week_of_year|day_of_month|day_of_week|day_name|day_abbr|is_weekend|is_leap_year|year_month|year_quarter|
+--------+----------+----+-------+-----+----------+----------+------------+------------+-----------+--------+--------+----------+------------+----------+------------+
|20240101|2024-01-01|2024|      1|    1|   January|       Jan|           1|           1|          2|  Monday|     Mon|     false|        true|   2024-01|     2024-Q1|
|20240704|2024-07-04|2024|      3|    7|      July|       Jul|          27|           4|          5|Thursday|     Thu|     false|        true|   2024-07|     2024-Q3|
|20241231|2024-12-31|2024|      4|   12|  December|       Dec|           1|          31|          3| Tuesday|     Tue|     false|        